In [1]:
import numpy as np
import pandas as pd
from sklift.datasets import fetch_hillstrom
from sklearn.model_selection import train_test_split

from sklift.metrics import qini_auc_score, uplift_at_k

import matplotlib.pyplot as plt

from QuiniDeep import *
from SMITE import *
from GNUM import *
from DragonNet import *

/Users/romanseleznyov/Documents/UpliftNinja/uplift-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

In [3]:
%%time

data = fetch_hillstrom(target_col='visit')
df = data.data
X, y, w = df, data.target, data.treatment

# Hillstrom: treatment — 'Mens E-Mail', 'Womens E-Mail' -> объединим в бинарный treatment
w = (w != 'No E-Mail').astype(int)  # 1 — получили email, 0 — нет

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=SEED
)

num_cols = X_train.select_dtypes(exclude='object').columns
X_train = X_train[num_cols]
X_test = X_test[num_cols]

# for causalml
X_np = X[num_cols].values
y_np = y.values
X_train_np, X_test_np, y_train_np, y_test_np, w_train_str, w_test_str = (
    X_train.values, X_test.values,
    y_train.values, y_test.values,
    np.where(w_train == 1, 'treatment', 'control'),
    np.where(w_test == 1, 'treatment', 'control')
)
w_train_np = w_train.to_numpy()
w_test_np = w_test.to_numpy()

CPU times: user 54.2 ms, sys: 13.7 ms, total: 67.9 ms
Wall time: 68.2 ms


In [4]:
cfg = DeepUpliftNetConfig(
    n_features=X_train.shape[1],
    n_treatments=2,          # control + 2 treatments
    outcome_type="binary",   # heads output logits
    parameterization="direct"
)

In [5]:
model = DeepUpliftNetwork(cfg)

In [6]:
train_ds = UpliftDataset(X_train_np, w_train_np, y_train_np)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

valid_ds = UpliftDataset(X_test_np, w_test_np, y_test_np)
valid_loader = DataLoader(valid_ds, batch_size=2048, shuffle=False)

In [7]:
%%time

train_deep_uplift_network(model, train_loader, valid_loader=valid_loader)

Epoch 001/100 | train_loss=0.798986 | valid_loss=0.440666
Epoch 002/100 | train_loss=0.447371 | valid_loss=0.419118
Epoch 003/100 | train_loss=0.431294 | valid_loss=0.420393
Epoch 004/100 | train_loss=0.422628 | valid_loss=0.420084
Epoch 005/100 | train_loss=0.414010 | valid_loss=0.402878
Epoch 006/100 | train_loss=0.415323 | valid_loss=0.405483
Epoch 007/100 | train_loss=0.414139 | valid_loss=0.402411
Epoch 008/100 | train_loss=0.411836 | valid_loss=0.404414
Epoch 009/100 | train_loss=0.409837 | valid_loss=0.405922
Epoch 010/100 | train_loss=0.412871 | valid_loss=0.424357
Epoch 011/100 | train_loss=0.409649 | valid_loss=0.403016
Epoch 012/100 | train_loss=0.409693 | valid_loss=0.403148
Epoch 013/100 | train_loss=0.407912 | valid_loss=0.405958
Epoch 014/100 | train_loss=0.407676 | valid_loss=0.405130
Epoch 015/100 | train_loss=0.408953 | valid_loss=0.401041
Epoch 016/100 | train_loss=0.407186 | valid_loss=0.406368
Epoch 017/100 | train_loss=0.407608 | valid_loss=0.404010
Epoch 018/100 

{'train_loss': [0.798986182808876,
  0.44737102746963503,
  0.43129357397556306,
  0.4226275637745857,
  0.4140098088979721,
  0.4153228971362114,
  0.41413901656866076,
  0.41183600038290025,
  0.4098365956544876,
  0.4128714886307716,
  0.4096486982703209,
  0.4096926811337471,
  0.4079120445251465,
  0.4076756536960602,
  0.4089528861641884,
  0.4071858471632004,
  0.4076075550913811,
  0.4068659344315529,
  0.40711479157209396,
  0.40728681117296217,
  0.4090000197291374,
  0.4063392984867096,
  0.4071077796816826,
  0.4069744578003883,
  0.40597402811050415,
  0.4069456565380096,
  0.4063906866312027,
  0.4059694600105286,
  0.40707202136516574,
  0.4066699945926666,
  0.4062500748038292,
  0.40609807163476946,
  0.4056748604774475,
  0.4069501584768295,
  0.40759206712245943,
  0.4060222002863884,
  0.4062376222014427,
  0.40660672545433046,
  0.4063723537325859,
  0.40694543182849885,
  0.4063220486044884,
  0.40635196983814237,
  0.4060016542673111,
  0.40717292368412017,
  0.4

In [8]:
tau_hat = model.predict_uplift(X_test_np)[:, 0]   # (n, 2) — удобно отдавать в maq/свои метрики

In [9]:
qini_auc_score(y_test_np, tau_hat, w_test_np)

np.float64(0.02902076134786732)

In [10]:
uplift_at_k(y_test_np, tau_hat, w_test_np, strategy='by_group', k=0.3)

np.float64(0.07890749783233697)

# SMITE(TO)

In [13]:
cfg = SMITEConfig(
    n_features=X_train.shape[1],
    uplift_loss="to",
    propensity_e=0.5,   # RCT 50/50
    alpha=0.5
)
model = SMITE(cfg)

In [16]:
train_ds = SMITEDataset(X_train_np, w_train_np, y_train_np)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

valid_ds = SMITEDataset(X_test_np, w_test_np, y_test_np)
valid_loader = DataLoader(valid_ds, batch_size=4096, shuffle=False)

In [17]:
%%time

train_smite(model, train_loader, valid_loader=valid_loader)

/Users/romanseleznyov/Documents/UpliftNinja/upninja/torch/SMITE.py:448: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(train_cfg.use_amp and device.type == "cuda"))
/Users/romanseleznyov/Documents/UpliftNinja/upninja/torch/SMITE.py:470: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):


Epoch 001/50 | train_loss=0.639222 | valid_loss=0.509777
Epoch 002/50 | train_loss=0.513208 | valid_loss=0.496389
Epoch 003/50 | train_loss=0.510526 | valid_loss=0.495362
Epoch 004/50 | train_loss=0.504222 | valid_loss=0.486712
Epoch 005/50 | train_loss=0.501771 | valid_loss=0.503902
Epoch 006/50 | train_loss=0.498041 | valid_loss=0.482069
Epoch 007/50 | train_loss=0.502609 | valid_loss=0.520869
Epoch 008/50 | train_loss=0.502953 | valid_loss=0.482447
Epoch 009/50 | train_loss=0.495073 | valid_loss=0.479299
Epoch 010/50 | train_loss=0.500487 | valid_loss=0.496452
Epoch 011/50 | train_loss=0.509166 | valid_loss=0.478964
Epoch 012/50 | train_loss=0.494433 | valid_loss=0.483977
Epoch 013/50 | train_loss=0.496459 | valid_loss=0.478599
Epoch 014/50 | train_loss=0.497290 | valid_loss=0.478229
Epoch 015/50 | train_loss=0.495237 | valid_loss=0.481623
Epoch 016/50 | train_loss=0.492552 | valid_loss=0.484018
Epoch 017/50 | train_loss=0.490156 | valid_loss=0.477014
Epoch 018/50 | train_loss=0.492

{'train_loss': [0.6392222836613655,
  0.5132080048322678,
  0.510526137650013,
  0.5042221495509147,
  0.5017713907361031,
  0.4980409380793571,
  0.5026085469126701,
  0.5029528135061264,
  0.495073499083519,
  0.5004870709776879,
  0.5091659194231033,
  0.49443272382020953,
  0.49645854651927945,
  0.4972899225354195,
  0.49523687541484834,
  0.4925523439049721,
  0.49015614837408067,
  0.49269906789064405,
  0.4948253428936005,
  0.496855206489563,
  0.4929897439479828,
  0.4909426146745682,
  0.4915517446398735,
  0.4945699781179428,
  0.4907401233911514,
  0.4919291844964027,
  0.490448679625988,
  0.4896005117893219,
  0.49122648775577543,
  0.49746447294950485,
  0.4911094635725021,
  0.489069844186306,
  0.4904833310842514,
  0.490967378616333,
  0.4923467618227005,
  0.4896046182513237,
  0.4904709926247597,
  0.4931396198272705,
  0.48991414308547976,
  0.4901568168401718,
  0.48945548295974733,
  0.4898778909444809,
  0.4888421532511711,
  0.49203790336847303,
  0.4907986885

In [18]:
uplift = model.predict_uplift(X_test_np)

In [19]:
qini_auc_score(y_test_np, uplift, w_test_np)

np.float64(0.009083137581759462)

In [20]:
uplift_at_k(y_test_np, uplift, w_test_np, strategy='by_group', k=0.3)

np.float64(0.06513941495915665)

# SMITE(IE)

In [21]:
cfg = SMITEConfig(
    n_features=X_train.shape[1],
    uplift_loss="ie",
    alpha=0.5
)
model = SMITE(cfg)

In [22]:
train_ds = SMITEDataset(X_train_np, w_train_np, y_train_np)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

valid_ds = SMITEDataset(X_test_np, w_test_np, y_test_np)
valid_loader = DataLoader(valid_ds, batch_size=4096, shuffle=False)

In [23]:
%%time

train_smite(model, train_loader, valid_loader=valid_loader)

/Users/romanseleznyov/Documents/UpliftNinja/upninja/torch/SMITE.py:448: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(train_cfg.use_amp and device.type == "cuda"))
/Users/romanseleznyov/Documents/UpliftNinja/upninja/torch/SMITE.py:470: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):


Epoch 001/50 | train_loss=0.938278 | valid_loss=0.563399
Epoch 002/50 | train_loss=0.572099 | valid_loss=0.561443
Epoch 003/50 | train_loss=0.561755 | valid_loss=0.557044
Epoch 004/50 | train_loss=0.559472 | valid_loss=0.554284
Epoch 005/50 | train_loss=0.559292 | valid_loss=0.553405
Epoch 006/50 | train_loss=0.559843 | valid_loss=0.556516
Epoch 007/50 | train_loss=0.556509 | valid_loss=0.551984
Epoch 008/50 | train_loss=0.557068 | valid_loss=0.551416
Epoch 009/50 | train_loss=0.552764 | valid_loss=0.549460
Epoch 010/50 | train_loss=0.553242 | valid_loss=0.547694
Epoch 011/50 | train_loss=0.553245 | valid_loss=0.546912
Epoch 012/50 | train_loss=0.555033 | valid_loss=0.548944
Epoch 013/50 | train_loss=0.550457 | valid_loss=0.547071
Epoch 014/50 | train_loss=0.552730 | valid_loss=0.547553
Epoch 015/50 | train_loss=0.550988 | valid_loss=0.555411
Epoch 016/50 | train_loss=0.551449 | valid_loss=0.545885
Epoch 017/50 | train_loss=0.551018 | valid_loss=0.554070
Epoch 018/50 | train_loss=0.550

{'train_loss': [0.9382779228687287,
  0.5720990645885468,
  0.5617547470331192,
  0.5594718128442764,
  0.5592919301986694,
  0.5598427748680115,
  0.5565092009305954,
  0.5570676410198212,
  0.5527644520998001,
  0.5532423150539398,
  0.5532454890012741,
  0.5550327003002167,
  0.5504570424556732,
  0.5527296888828278,
  0.5509884613752365,
  0.551448962688446,
  0.551018041074276,
  0.5500716942548752,
  0.5530060428380966,
  0.5497052586078643,
  0.5503963679075241,
  0.5492816889286041,
  0.5507960098981858,
  0.5492873519659043,
  0.5605289506912231,
  0.5488513994216919,
  0.548667414188385,
  0.5496482735872269,
  0.5505380690097809,
  0.548433244228363,
  0.5494845163822174,
  0.549019119143486,
  0.5491433364152908,
  0.5484916579723358,
  0.548672097325325,
  0.5481588745117187,
  0.5500709593296051,
  0.5500821655988694,
  0.550217952132225,
  0.548099011182785,
  0.5479818409681321,
  0.5478351646661759,
  0.5490703678131104,
  0.5473962992429733,
  0.5492159384489059,
  0.

In [24]:
uplift = model.predict_uplift(X_test_np)

In [25]:
qini_auc_score(y_test_np, uplift, w_test_np)

np.float64(0.031384455518864686)

In [26]:
uplift_at_k(y_test_np, uplift, w_test_np, strategy='by_group', k=0.3)

np.float64(0.08357078750817626)

# DragonNet

In [4]:
# Конфиг модели
m_cfg = DragonNetConfig(
    n_features=X_train.shape[1],
    outcome_type="binary",                 # или "continuous"
    use_targeted_regularization=True,
    lambda_propensity=1.0,
    lambda_targeted=1.0,
)

model = DragonNet(m_cfg)

In [5]:
# Датасеты/лоадеры
train_ds = DragonNetDataset(X_train_np, w_train_np, y_train_np)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

valid_ds = DragonNetDataset(X_test_np, w_test_np, y_test_np)
valid_loader = DataLoader(valid_ds, batch_size=4096, shuffle=False)

In [6]:
%%time
# Тренировка
train_dragonnet(model, train_loader, valid_loader=valid_loader)

/Users/romanseleznyov/Documents/UpliftNinja/upninja/torch/DragonNet.py:430: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))
/Users/romanseleznyov/Documents/UpliftNinja/upninja/torch/DragonNet.py:448: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):


Epoch 001/100 | train_loss=88.765511 | valid_loss=1.184322
Epoch 002/100 | train_loss=1.192724 | valid_loss=1.167575
Epoch 003/100 | train_loss=1.176244 | valid_loss=1.162404
Epoch 004/100 | train_loss=1.172296 | valid_loss=1.157305
Epoch 005/100 | train_loss=1.169563 | valid_loss=1.158679
Epoch 006/100 | train_loss=1.169670 | valid_loss=1.157824
Epoch 007/100 | train_loss=1.168359 | valid_loss=1.162818
Epoch 008/100 | train_loss=1.170901 | valid_loss=1.158946
Epoch 009/100 | train_loss=1.169418 | valid_loss=1.157437
Epoch 010/100 | train_loss=1.168762 | valid_loss=1.156406
Epoch 011/100 | train_loss=1.168823 | valid_loss=1.159919
Epoch 012/100 | train_loss=1.167839 | valid_loss=1.155309
Epoch 013/100 | train_loss=1.168431 | valid_loss=1.156268
Epoch 014/100 | train_loss=1.168358 | valid_loss=1.158163
Epoch 015/100 | train_loss=1.167800 | valid_loss=1.155011
Epoch 016/100 | train_loss=1.169255 | valid_loss=1.157993
Epoch 017/100 | train_loss=1.167199 | valid_loss=1.154020
Epoch 018/100

{'train_loss': [88.7655113852024,
  1.1927236258983611,
  1.1762443447113038,
  1.172295982837677,
  1.1695627117156981,
  1.1696699059009552,
  1.16835902094841,
  1.1709005808830262,
  1.1694179952144623,
  1.1687616610527038,
  1.1688228678703307,
  1.1678389823436737,
  1.1684306788444518,
  1.1683580422401427,
  1.1677998089790345,
  1.1692547988891602,
  1.1671991074085235,
  1.1667094659805297,
  1.1679609775543214,
  1.1679104602336883,
  1.1665584421157837,
  1.1668894791603088,
  1.1688966631889344,
  1.166695886850357,
  1.1652510893344878,
  1.1681104433536529,
  1.1659900033473969,
  1.1675574088096619,
  1.1675688314437866,
  1.1665568411350251,
  1.166314549446106,
  1.1664060986042022,
  1.1663889062404633,
  1.1658467078208923,
  1.166521829366684,
  1.165641417503357,
  1.1673807203769684,
  1.1658081197738648,
  1.16593421459198,
  1.1660463893413544,
  1.1660679495334625,
  1.1656571733951568,
  1.1644761919975282,
  1.1646021974086762,
  1.1647394049167632,
  1.166

In [7]:
# Инференс uplift
tau_hat = model.predict_uplift(X_test_np)

In [8]:
# ---- Sanity checks ----
assert tau_hat.shape[0] == X_test_np.shape[0]
po = model.predict_potential_outcomes(X_test_np)
assert po.shape == (X_test_np.shape[0], 2)  # [Y0, Y1]

In [9]:
qini_auc_score(y_test_np, tau_hat, w_test_np)

np.float64(0.028342818915170955)

In [11]:
uplift_at_k(y_test_np, tau_hat, w_test_np, strategy='by_group', k=0.3)

np.float64(0.08818296597149333)